# Center and spread: mean, median and standard deviation

Core, Part 1 (Statistics I), lesson C2. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** compute mean, median and mode and say when each misleads, compute range, IQR and standard deviation, read percentiles and a box plot, and write a two-sentence summary of a column.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each completed order, with its total in dollars: quantity × price − discount, summed over the order's lines.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(amount=order_items["quantity"] * order_items["product_id"].map(price) - order_items["discount_usd"])
order_total = lines.groupby("order_id")["amount"].sum().rename("total")
sales = orders[orders["status"] == "completed"].merge(order_total, on="order_id", validate="one_to_one")
totals = sales["total"]
assert len(sales) == 1799  # the canceled order of 5.00 is left out
print("1,799 completed orders. `totals` holds the total of each one, in dollars.")

## Unit 1: Three centers

**Predict first.** will the mean of the order totals be above or below the median? And what is the mode?

In [ ]:
print(round(totals.mean(), 2), totals.median(), totals.mode().tolist())

**Predict first.** the 11 orders of May 1 hold one order of 197.00, the line of 20 units that the README says should read 2. Drop that order: how far do the mean and the median of the day move? And how far do the mean and the median of all 1,799 orders move when the same order is dropped?

In [ ]:
day = sales.loc[sales["ordered_at"].str.startswith("2026-05-01"), "total"]
rest = day.drop(day.idxmax())
print(len(day), round(day.mean(), 2), day.median())
print(round(rest.mean(), 2), rest.median())
print(round(totals.mean() - totals.drop(totals.idxmax()).mean(), 2), totals.median(), totals.drop(totals.idxmax()).median())

## Unit 2: Three spreads

**Predict first.** what are the range, the IQR and the standard deviation of the 1,799 totals?

In [ ]:
q1, q3 = totals.quantile(0.25), totals.quantile(0.75)
print(totals.max() - totals.min(), q3 - q1, round(totals.std(), 2))

**Predict first.** drop the order of 197.00 and compute the same three numbers. Which one barely moves?

In [ ]:
rest = totals.drop(totals.idxmax())
r1, r3 = rest.quantile(0.25), rest.quantile(0.75)
print(rest.max() - rest.min(), r3 - r1, round(rest.std(), 2))

**Predict first.** pandas and NumPy each have a standard deviation. On the 11 orders of May 1, will `day.std()` and `np.std(day)` agree? And on all 1,799 orders?

In [ ]:
print(round(day.std(), 2), round(np.std(day), 2))
print(round(totals.std(), 4), round(np.std(totals), 4))

## Unit 3: Percentiles and the box plot

**Predict first.** what are the 10th, 25th, 50th, 75th and 90th percentiles of the totals?

In [ ]:
print(totals.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).tolist())

**Predict first.** the box plot draws a dot for each order above the fence at the 75th percentile plus 1.5 times the IQR. Where is that fence, and how many dots will there be?

In [ ]:
fence = q3 + 1.5 * (q3 - q1)
fig, ax = plt.subplots()
bp = ax.boxplot(totals)
ax.set_title("Order totals, January to June 2026")
ax.set_ylabel("Order total (USD)")
print(fence, len(bp["fliers"][0].get_ydata()), (totals > fence).sum())

## Unit 4: Say it in two sentences

**Predict first.** which two numbers will the summary give for the center and the spread of a column like this one?

In [ ]:
q1, med, q3 = totals.quantile([0.25, 0.5, 0.75])
summary = (f"Half of the {len(totals):,} completed orders cost between {q1:.2f} and {q3:.2f} dollars, and the middle order costs {med:.2f}. "
           f"The mean, {totals.mean():.2f}, sits above the median because a few large orders pull it up; the largest, {totals.max():.2f}, is worth a check.")
print(summary)

## Finish it

### Exercise 1: Finish it

**Task:** What is the 90th percentile of the order totals at Campus, store 2? That is the total that 9 in 10 Campus orders do not exceed. Put the dollars in `answer_1`, rounded to 2 decimals.

**Expected:** `answer_1` is a number in dollars.

In [ ]:
campus = sales[sales["store_id"] == 2]["total"]  # given: the totals of the Campus orders
# find the 90th percentile of campus, and set answer_1
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 22.75, traps={11.5: "That is the median, the 50th percentile. Which fraction does `quantile` take for the 90th?"})

<details>
<summary>Hint</summary>

`quantile` takes a fraction between 0 and 1, not a percent.

</details>

<details>
<summary>Solution</summary>

```python
campus = sales[sales["store_id"] == 2]["total"]
answer_1 = round(float(campus.quantile(0.9)), 2)
```

22.75 dollars: nine in ten Campus orders are 22.75 or less. `quantile(0.5)` would be the median, 11.50, which the check cell reports if you pass the wrong fraction.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many Harbor orders are unusually large for Harbor? Before you start, say in one sentence what the result looks like: one whole number. Use the fence of the box plot: the 75th percentile of the Harbor totals plus 1.5 times the IQR of the Harbor totals, and count the Harbor orders above it. Harbor is store 3. Put the count in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 10, traps={44: "That counts the orders of all three stores above the Harbor fence. Count only the Harbor orders."})

<details>
<summary>Hint</summary>

Take the Harbor totals first; every number after that, the quartiles, the fence and the count, comes from them.

</details>

<details>
<summary>Solution</summary>

```python
harbor = sales[sales["store_id"] == 3]["total"]
h1, h3 = harbor.quantile(0.25), harbor.quantile(0.75)
fence_harbor = h3 + 1.5 * (h3 - h1)
answer_2 = int((harbor > fence_harbor).sum())
```

Harbor's quartiles are 7.50 and 17.00, so its fence is 31.25, a little above the 30.75 of all orders together; 10 of the 355 Harbor orders are above it. Counting the orders of all three stores above that fence gives 44, which the check cell reports.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague reports the share of cash orders for the whole chain as the mean of the three store shares, because each store's share is already computed (a share is the mean of a column of True and False). Set `answer_3` to the share of cash orders among all 1,799 completed orders in `sales`, as a number between 0 and 1, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
store_shares = sales.groupby("store_id")["payment_method"].apply(lambda s: (s == "cash").mean())  # the colleague's three store shares
answer_3 = round(float(store_shares.mean()), 3)  # the colleague's average of them

In [ ]:
_ = check("Exercise 3", "answer_3", 0.14, traps={0.143: "That is the mean of the three store shares. The stores took different numbers of orders. Which share counts every order once?"})

<details>
<summary>Hint</summary>

Harbor took about half as many orders as each of the other two stores. An average of three shares counts a Harbor order twice as heavily.

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = round(float((sales["payment_method"] == "cash").mean()), 3)
```

0.140: 252 of the 1,799 orders were paid in cash. The mean of the three store shares is 0.143, because Harbor, with 355 orders, counts as much as Old Town with 729 or Campus with 715. The check cell reports it.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")